In [ ]:
# Imports
import os
from pathlib import Path
markers = (".git", "Program")
current_dir = Path.cwd()
project_root = next((path for path in (current_dir, *current_dir.parents) if any((path / marker).exists() for marker in markers)), current_dir)
os.chdir(project_root)

from backtest import *
import concurrent.futures
import datetime as dt
from functools import partial
from fundamentals import *
from helper_functions import modify_current_date, get_df, get_excel_filename, get_infix, get_volume5m_data, generate_end_dates, merge_stocks, stock_market
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
import multiprocessing
import numpy as np
import pandas as pd
from pandas import ExcelWriter as EW
from plot import *
import random
from scipy.optimize import minimize
from scipy.stats import false_discovery_control, kendalltau, linregress, pearsonr, spearmanr, ttest_ind, wilcoxon
from scipy.cluster.hierarchy import linkage, dendrogram, fcluster
from scipy.spatial.distance import squareform
from stock_screener import check_conds_tech, check_conds_fund, EM_rating, get_stock_info, stoploss_target
from technicals import *
from tqdm import tqdm

# Connect to TradingView
from tvDatafeed import TvDatafeed, Interval
tv = TvDatafeed()

# Start of the program
start = dt.datetime.now()

# Index
index_name = "^GSPC"
index_dict = {"^HSI": "HKEX", "^GSPC": "S&P 500", "^IXIC": "NASDAQ Composite"}

# Modify the current date
current_date = modify_current_date(start, index_name)

In [ ]:
def get_parameters():
    """Return default parameters for stock analysis.
    
    Returns:
    dict: Configuration parameters including:
    - days_per_week: Trading days per week
    - period_week_zscore: Lookback period for weekly z-score calculation
    - period_pca: Period for correlation/clustering analysis
    - period_mom_short/long: Short and long-term momentum periods
    - period_vol: Volatility calculation period
    - num_clusters: Number of clusters for hierarchical clustering
    """
    days_per_week = 5
    num_weeks = 52
    return {
        "days_per_week": days_per_week,
        "period_week_zscore": days_per_week * num_weeks,
        "period_pca": 126,
        "period_mom_short": 21,
        "period_mom_long": 252,
        "period_vol": 60,
        "num_clusters": 5,
    }

def fetch_price_data(stocks, current_date):
    """Fetch historical price data for all stocks.
    
    Args:
    - stocks: List of stock tickers
    - current_date: End date for data retrieval
        
    Returns:
        dict: Mapping of stock ticker to closing price series
    """
    price_data = {}
    for stock in tqdm(stocks, desc="Fetching price data"):
        df = get_df(stock, current_date)
        price_data[stock] = df["Close"]
    return price_data

def calculate_weekly_zscore(price_data, stocks, params):
    """Calculate weekly return z-scores to identify statistical outliers.
    
    Z-score measures how many standard deviations the recent weekly return
    is from the historical mean. High z-scores indicate unusually strong returns.
    
    Args:
    - price_data: Dict of stock ticker to price series
    - stocks: List of stock tickers
    - params: Parameters dict containing period settings
        
    Returns:
        tuple: (mean_return, std_return, recent_return, z_scores) as Series
    """
    df_prices_weekly = pd.DataFrame({
        stock: price_data[stock].tail(params["period_week_zscore"] + params["days_per_week"]) 
        for stock in stocks
    })
    weekly_prices = df_prices_weekly.iloc[::params["days_per_week"]]
    weekly_returns = weekly_prices.pct_change(fill_method=None).dropna()
    
    mean_return = weekly_returns.mean()
    std_return = weekly_returns.std()
    recent_return = weekly_returns.iloc[-1]
    z_scores = (recent_return - mean_return) / std_return
    
    return mean_return, std_return, recent_return, z_scores

def calculate_momentum_volatility(price_data, stocks, params):
    """Calculate momentum and volatility metrics for all stocks.
    
    Momentum: Ratio of short-term to long-term price (higher = stronger trend)
    Volatility: Standard deviation of daily returns over the vol period
    
    Args:
    - price_data: Dict of stock ticker to price series
    - stocks: List of stock tickers
    - params: Parameters dict containing period settings
        
    Returns:
    - tuple: (momentum_list, volatility_list) as lists aligned with stocks
    """
    momentum_list = []
    volatility_list = []
    
    for stock in tqdm(stocks, desc="Calculating momentum and volatility"):
        close = price_data[stock]
        momentum = close.iloc[-params["period_mom_short"]] / close.iloc[-params["period_mom_long"]] if len(close) >= params["period_mom_long"] else np.nan
        volatility = close.pct_change().tail(params["period_vol"]).std()
        momentum_list.append(momentum)
        volatility_list.append(volatility)
    
    return momentum_list, volatility_list

def perform_clustering(price_data, stocks, params):
    """Perform hierarchical clustering on stocks based on return correlations.
    
    Uses Ward's method on a distance matrix derived from correlation matrix.
    Stocks with similar return patterns are grouped into the same cluster.
    
    Args:
    - price_data: Dict of stock ticker to price series
    - stocks: List of stock tickers
    - params: Parameters dict containing period and cluster settings
        
    Returns:
        tuple: (Z linkage matrix, cluster_labels array)
    """
    df_prices_cluster = pd.DataFrame({stock: price_data[stock].tail(params["period_pca"]) for stock in stocks})
    returns = df_prices_cluster.pct_change(fill_method=None).dropna()
    corr_matrix = returns.corr()
    corr_matrix = corr_matrix.fillna(0)
    
    # Convert correlation to distance: uncorrelated = sqrt(2), perfectly correlated = 0
    dist_matrix = np.sqrt(2 * (1 - corr_matrix))
    dist_matrix = np.nan_to_num(dist_matrix, nan=0.0, posinf=2.0, neginf=0.0)
    condensed_dist = squareform(dist_matrix, checks=False)
    Z = linkage(condensed_dist, method="ward")
    cluster_labels = fcluster(Z, t=params["num_clusters"], criterion="maxclust")
    return Z, cluster_labels

def build_combined_dataframe(stocks, cluster_labels, momentum_list, volatility_list, mean_return, std_return, recent_return, z_scores):
    """Combine all computed metrics into a single ranked DataFrame.
    
    Calculates volatility-adjusted momentum (Momentum / Volatility) to identify
    stocks with strong trends relative to their risk level.
    
    Args:
    - stocks: List of stock tickers
    - cluster_labels: Cluster assignment for each stock
    - momentum_list: Momentum values for each stock
    - volatility_list: Volatility values for each stock
    - mean_return: Mean weekly return series
    - std_return: Std of weekly return series
    - recent_return: Most recent weekly return series
    - z_scores: Z-score series
        
    Returns:
    - DataFrame: Ranked by Vol-adj Momentum with 1-based index
    """
    df_combined = pd.DataFrame({
        "Stock": stocks,
        "Cluster": cluster_labels,
        "Momentum": momentum_list,
        "Volatility": volatility_list,
        "Mean Weekly Return (%)": (mean_return * 100).values,
        "Std Weekly Return (%)": (std_return * 100).values,
        "This Week Return (%)": (recent_return * 100).values,
        "Z-Score": z_scores.values
    })
    df_combined["Vol-adj Momentum"] = df_combined["Momentum"] / df_combined["Volatility"]
    df_combined = df_combined.sort_values("Vol-adj Momentum", ascending=False).reset_index(drop=True)
    df_combined = df_combined.set_index(df_combined.index + 1)
    df_combined.index.name = "Rank"
    return df_combined

def display_results(df_combined, Z, stocks, params, zscore_threshold=2):
    """Display analysis results including table, dendrogram, and cluster summary.
    
    Filters stocks by z-score threshold to exclude recent outliers,
    then groups remaining stocks by cluster for diversified selection.
    
    Args:
    - df_combined: Combined metrics DataFrame
    - Z: Linkage matrix for dendrogram
    - stocks: List of stock tickers
    - params: Parameters dict
    - zscore_threshold: Max z-score for filtered display (default 2)
    """
    pd.set_option("display.max_rows", None)
    display(df_combined)
    
    plt.figure(figsize=(14, 8))
    dendrogram(Z, labels=stocks, leaf_rotation=90, leaf_font_size=8)
    plt.title(f"Hierarchical Clustering of {len(stocks)} Stocks (Past {params['period_pca']} Days)")
    plt.ylabel("Distance")
    plt.tight_layout()
    plt.show()
    
    df_filtered = df_combined[df_combined["Z-Score"] <= zscore_threshold]
    print(f"\nStocks by Cluster (Ranked by Vol-adj Momentum, Z-Score <= {zscore_threshold}):\n")
    for cluster_id in sorted(df_filtered["Cluster"].unique()):
        cluster_df = df_filtered[df_filtered["Cluster"] == cluster_id].sort_values("Vol-adj Momentum", ascending=False)
        cluster_stocks = cluster_df["Stock"].tolist()
        print(f"Cluster {cluster_id} ({len(cluster_df)} stocks): {', '.join(cluster_stocks)}")

def run_stock_analysis(stocks, current_date, display=True):
    """Run the complete stock analysis pipeline.
    
    Pipeline steps:
    1. Fetch price data for all stocks
    2. Calculate weekly return z-scores
    3. Calculate momentum and volatility
    4. Perform hierarchical clustering
    5. Build combined metrics DataFrame
    6. Optionally display results
    
    Args:
    - stocks: List of stock tickers to analyze
    - current_date: Analysis date (end date for data)
    - display: Whether to show results (default True)
        
    Returns:
    - DataFrame: Combined metrics ranked by Vol-adj Momentum
    """
    params = get_parameters()
    price_data = fetch_price_data(stocks, current_date)
    mean_return, std_return, recent_return, z_scores = calculate_weekly_zscore(price_data, stocks, params)
    momentum_list, volatility_list = calculate_momentum_volatility(price_data, stocks, params)
    Z, cluster_labels = perform_clustering(price_data, stocks, params)
    df_combined = build_combined_dataframe(stocks, cluster_labels, momentum_list, volatility_list, mean_return, std_return, recent_return, z_scores)
    if display:
        display_results(df_combined, Z, stocks, params)
    return df_combined

In [ ]:
# Rebalance dates for monthly portfolio analysis
rebalance_dates = [
    "2025-01-04", "2025-02-01", "2025-03-01", "2025-04-05", "2025-05-01", "2025-06-03",
    "2025-07-01", "2025-08-01", "2025-09-06", "2025-10-01", "2025-11-01", "2025-12-04", "2026-01-03"
]

# Parameters
min_market_cap = 10  # Minimum market cap in billions USD
lookback_period = 252
volume_period = 90

# Extract stocks meeting market cap criteria for each rebalance date
monthly_stocks = {}
for date in rebalance_dates:
    excel_filename = get_excel_filename(date, index_name, index_dict, lookback_period, volume_period, True)
    excel_df = pd.read_excel(excel_filename)
    
    # Filter stocks with market cap above threshold
    stocks_df = excel_df.loc[excel_df["Market Cap (B, USD)"] > min_market_cap, ["Stock", "Market Cap (B, USD)"]]
    stocks = stocks_df["Stock"].tolist()
    monthly_stocks[date] = stocks
    print(f"{date}: {len(stocks)} stocks from {excel_filename}")

# Run stock analysis and build portfolios for each rebalance date
monthly_portfolios = {}

for date in rebalance_dates:
    print(f"Rebalance Date: {date}")
    
    # Run clustering and momentum analysis
    stocks = monthly_stocks[date]
    df_combined = run_stock_analysis(stocks, date, display=False)
    
    # Select top stock from each cluster (highest vol-adjusted momentum)
    top_stocks_per_cluster = df_combined.groupby("Cluster").first().reset_index()
    
    # Calculate inverse volatility weights for risk parity allocation
    top_stocks = top_stocks_per_cluster["Stock"].tolist()
    volatilities = top_stocks_per_cluster["Volatility"].tolist()
    
    inv_vol = [1 / v for v in volatilities]
    total_inv_vol = sum(inv_vol)
    weights = [iv / total_inv_vol for iv in inv_vol]
    
    # Build and store portfolio allocation
    result_df = pd.DataFrame({
        "Cluster": top_stocks_per_cluster["Cluster"],
        "Stock": top_stocks,
        "Volatility": volatilities,
        "Weight (%)": [w * 100 for w in weights]
    })
    monthly_portfolios[date] = result_df
    display(result_df)

In [ ]:
# Backtest the monthly portfolios with daily equity curve
sp500_df = get_df("^GSPC", current_date)

# Initialize daily equity tracking
daily_equity = []
initial_value = 100.0

# Iterate through each rebalancing period
for i in range(len(rebalance_dates) - 1):
    start_date = rebalance_dates[i]
    end_date = rebalance_dates[i + 1]
    
    # Skip if no portfolio exists for this period
    if start_date not in monthly_portfolios:
        continue
    
    portfolio = monthly_portfolios[start_date]
    
    # Find buy date: first trading day after rebalance date
    start_dt = pd.to_datetime(start_date)
    valid_dates = sp500_df.index[sp500_df.index > start_dt]
    if len(valid_dates) == 0:
        continue
    buy_date = valid_dates[0]
    
    # Find sell date: last trading day before next rebalance date
    end_dt = pd.to_datetime(end_date)
    valid_dates = sp500_df.index[sp500_df.index < end_dt]
    if len(valid_dates) == 0:
        continue
    sell_date = valid_dates[-1]
    
    # Get all trading days in this period
    period_dates = sp500_df.index[(sp500_df.index >= buy_date) & (sp500_df.index <= sell_date)]
    
    # Get buy prices (open price on buy date) for each stock
    buy_prices = {}
    for _, row in portfolio.iterrows():
        stock = row["Stock"]
        stock_df = get_df(stock, current_date)
        
        if buy_date in stock_df.index:
            buy_prices[stock] = stock_df.loc[buy_date, "Open"]
        else:
            # Use next available date if exact date not found
            buy_idx = stock_df.index.searchsorted(buy_date)
            if buy_idx < len(stock_df):
                buy_prices[stock] = stock_df.iloc[buy_idx]["Open"]
    
    # Calculate daily portfolio value for each trading day
    for trade_date in period_dates:
        daily_return = 0.0
        
        for _, row in portfolio.iterrows():
            stock = row["Stock"]
            weight = row["Weight (%)"] / 100
            
            if stock not in buy_prices:
                continue
            
            stock_df = get_df(stock, current_date)
            
            # Get close price for current date
            if trade_date in stock_df.index:
                close_price = stock_df.loc[trade_date, "Close"]
            else:
                # Use most recent available close price
                close_idx = stock_df.index.searchsorted(trade_date)
                if close_idx > 0:
                    close_price = stock_df.iloc[close_idx - 1]["Close"]
                else:
                    continue
            
            # Calculate weighted return contribution
            stock_return = (close_price - buy_prices[stock]) / buy_prices[stock]
            daily_return += weight * stock_return
        
        # Calculate portfolio value relative to period start
        portfolio_value = initial_value * (1 + daily_return)
        
        daily_equity.append({
            "Date": trade_date,
            "Portfolio Value": portfolio_value,
            "Period": i + 1
        })
    
    # Update initial value for next period (compound returns)
    if len(daily_equity) > 0:
        initial_value = daily_equity[-1]["Portfolio Value"]

# Create daily equity DataFrame
daily_equity_df = pd.DataFrame(daily_equity)
daily_equity_df.set_index("Date", inplace=True)

# Get S&P 500 equity curve for comparison (normalized to 100)
sp500_period = sp500_df.loc[daily_equity_df.index.min():daily_equity_df.index.max(), "Close"]
sp500_equity = 100 * sp500_period / sp500_period.iloc[0]

# Calculate daily returns for risk metrics
portfolio_daily_returns = daily_equity_df["Portfolio Value"].pct_change().dropna()
sp500_daily_returns = sp500_equity.pct_change().dropna()

# Risk-free rate assumptions
risk_free_rate = 0.03  # 3% annualized
daily_rf = risk_free_rate / 252  # Daily risk-free rate

def calculate_metrics(daily_returns, equity_curve):
    """
    Calculate key performance metrics for a given return series.
    
    Args:
        daily_returns: Series of daily returns
        equity_curve: Series of portfolio values over time
    
    Returns:
        Dictionary containing performance metrics
    """
    # Total and annualized return
    total_return = (equity_curve.iloc[-1] / equity_curve.iloc[0]) - 1
    n_days = len(equity_curve)
    annualized_return = (1 + total_return) ** (252 / n_days) - 1
    
    # Annualized volatility
    annualized_vol = daily_returns.std() * np.sqrt(252)
    
    # Sharpe Ratio: excess return per unit of total risk
    excess_returns = daily_returns - daily_rf
    sharpe = (excess_returns.mean() * 252) / annualized_vol
    
    # Sortino Ratio: excess return per unit of downside risk
    downside_returns = daily_returns[daily_returns < daily_rf]
    downside_std = downside_returns.std() * np.sqrt(252)
    sortino = (excess_returns.mean() * 252) / downside_std if downside_std > 0 else np.nan
    
    # Maximum Drawdown: largest peak-to-trough decline
    rolling_max = equity_curve.cummax()
    drawdown = (equity_curve - rolling_max) / rolling_max
    max_drawdown = drawdown.min()
    
    # Calmar Ratio: annualized return per unit of max drawdown
    calmar = annualized_return / abs(max_drawdown) if max_drawdown != 0 else np.nan
    
    return {
        "Total Return (%)": total_return * 100,
        "Annualized Return (%)": annualized_return * 100,
        "Annualized Volatility (%)": annualized_vol * 100,
        "Sharpe Ratio": sharpe,
        "Sortino Ratio": sortino,
        "Max Drawdown (%)": max_drawdown * 100,
        "Calmar Ratio": calmar
    }


# Calculate metrics for portfolio and benchmark
portfolio_metrics = calculate_metrics(portfolio_daily_returns, daily_equity_df["Portfolio Value"])
sp500_metrics = calculate_metrics(sp500_daily_returns, sp500_equity)

# Display comparison table
metrics_df = pd.DataFrame({
    "Portfolio": portfolio_metrics,
    "S&P 500": sp500_metrics
}).T
display(metrics_df)

# Create visualization
fig, axes = plt.subplots(2, 1, figsize=(14, 10))

# Plot 1: Equity curve comparison
axes[0].plot(daily_equity_df.index, daily_equity_df["Portfolio Value"], 
             linewidth=1.5, color='blue', label='Portfolio')
axes[0].plot(sp500_equity.index, sp500_equity.values, 
             linewidth=1.5, color='orange', label='S&P 500')

# Add vertical lines for rebalancing dates
for date in rebalance_dates[1:-1]:
    dt = pd.to_datetime(date)
    if daily_equity_df.index.min() <= dt <= daily_equity_df.index.max():
        axes[0].axvline(x=dt, color='gray', linestyle='--', alpha=0.3)

axes[0].set_xlabel("Date")
axes[0].set_ylabel("Portfolio Value (%)")
axes[0].set_title("Equity Curve Comparison (Starting at 100%)")
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Plot 2: Drawdown comparison
portfolio_rolling_max = daily_equity_df["Portfolio Value"].cummax()
portfolio_drawdown = (daily_equity_df["Portfolio Value"] - portfolio_rolling_max) / portfolio_rolling_max * 100

sp500_rolling_max = sp500_equity.cummax()
sp500_drawdown = (sp500_equity - sp500_rolling_max) / sp500_rolling_max * 100

axes[1].fill_between(portfolio_drawdown.index, portfolio_drawdown.values, 0, 
                     alpha=0.3, color='blue', label='Portfolio')
axes[1].fill_between(sp500_drawdown.index, sp500_drawdown.values, 0, 
                     alpha=0.3, color='orange', label='S&P 500')
axes[1].set_xlabel("Date")
axes[1].set_ylabel("Drawdown (%)")
axes[1].set_title("Drawdown Comparison")
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Print summary statistics
print(f"\nPortfolio vs S&P 500 Summary:")
print(f"{'Metric':<25} {'Portfolio':>15} {'S&P 500':>15}")
for metric in portfolio_metrics.keys():
    print(f"{metric:<25} {portfolio_metrics[metric]:>15.2f} {sp500_metrics[metric]:>15.2f}")